<a href="https://colab.research.google.com/github/sanmquin/AI/blob/main/src/Graphiko/Fetch-Business-Cluster-Videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fetch videos for business-cluster channels
This notebook reuses the Graphiko business-cluster lookup logic and then fetches Finder videos grouped by channel.


## Environment setup and data connections
This cell installs runtime dependencies and validates connectivity to MongoDB. The output confirms whether the notebook can read Finder collections before any analysis begins.


In [1]:
# Install runtime dependencies
!pip install -q pymongo dnspython pinecone pandas numpy matplotlib seaborn scikit-learn

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    # Retrieve the URI from Colab Secrets
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)

    # Send a ping to confirm connection
    client.admin.command('ping')
    print('✅ Successfully connected to MongoDB!')
except Exception as e:
    print(f'❌ MongoDB connection failed: {e}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 13.7 MB/s eta 0:00:00
✅ Successfully connected to MongoDB!


## Resolve the latest business cluster context
This cell opens Finder collections, detects the latest clustering version, and selects the business cluster anchor that defines the analysis universe.


In [2]:
# Access Finder DB collections and identify the latest business cluster
db = client['finder']
clusters_col = db['ChannelDescriptions_clusters']
items_col = db['ChannelDescriptions_items']
channels_col = db['channels']
videos_col = db['videos']

latest = clusters_col.find_one(sort=[('version', -1), ('createdAt', -1)])
latest_version = latest['version'] if latest else None
print('Latest cluster version:', latest_version)

if latest_version is None:
    print('No clusters found.')
    business_cluster = None
else:
    business_cluster = clusters_col.find_one({
        'version': latest_version,
        'name': {'$regex': '^business', '$options': 'i'}
    })

if business_cluster:
    business_cluster_id = business_cluster['_id']
    print('Business cluster found:', business_cluster.get('name'))
    print('Business cluster _id:', business_cluster_id)
else:
    business_cluster_id = None
    print('No business cluster found in the latest version.')


Latest cluster version: 2
Business cluster found: Business, Venture Capital, and Entrepreneurship
Business cluster _id: 69e41878685f30ed081ec5a6


## Expand the business cluster to channel metadata
This cell maps cluster items to channel records, producing channel IDs, channel titles, and descriptions used later for embeddings and reporting.


In [3]:
# Resolve channels associated with the business cluster
if business_cluster_id is None:
    print('Cannot fetch channels without a business cluster id.')
    channel_ids = []
    business_channels = []
else:
    item_docs = list(items_col.find(
        {'clusterId': business_cluster_id},
        {'_id': 0, 'textId': 1}
    ))
    channel_ids = [d['textId'] for d in item_docs]

    business_channels = list(channels_col.find(
        {'channelId': {'$in': channel_ids}},
        {'_id': 0, 'channelId': 1, 'title': 1, 'description': 1}
    ))

    print(f'Business cluster has {len(channel_ids)} channel ids and {len(business_channels)} channel docs.')
    for ch in sorted(business_channels, key=lambda x: x.get('title', ''))[:20]:
        print(f"- {ch.get('title', '(untitled)')} ({ch.get('channelId')})")

    if len(business_channels) > 20:
        print(f'... and {len(business_channels) - 20} more channels')


Business cluster has 27 channel ids and 27 channel docs.
- 20VC with Harry Stebbings (UCf0PBRjhf0rF8fWBIxTuoWA)
- ARK Invest (UCK-zlnUfoDHzUwXcbddtnkg)
- Alex Hormozi (UCUyDOdBWhC1MCxEjC46d-zw)
- All-In Podcast (UCESLZhusAkFfsNsApnjF_Cg)
- Anthony Pompliano (UCevXpeL8cNyAnww-NqJ4m2w)
- Asianometry (UC1LpsuAUaKoMzzJSEt5WImw)
- Bg2 Pod (UC-yRDvpR99LUc5l7i7jLzew)
- Bloomberg Originals (UCUMZ7gohGI9HcU9VNsr2FJQ)
- Dan Martell (UCA-mWX9CvCTVFWRMb9bKc9w)
- Garry Tan (UCIBgYfDjtWlbJhg--Z4sOgQ)
- Greg Isenberg (UCPjNBjflYl0-HQtUvOx0Ibw)
- Joe Lonsdale (UCJEDniyP_YtcsXikBELqicw)
- Lenny's Podcast (UC6t1O76G0jYXOAoYCm153dA)
- My First Million (UCyaN6mg5u8Cjy2ZI4ikWaug)
- Network State Podcast (UCKrpnfpTwncQ050VFXcVkuQ)
- Patrick Boyle (UCASM0cgfkJxQ1ICmRilfHLw)
- Peter H. Diamandis (UCvxm0qTrGN_1LMYgUaftWyQ)
- Principles by Ray Dalio (UCqvaXJ1K3HheTPNjH-KpwXQ)
- Real Vision Presents (UCBH5VZE_Y4F3CMcPIzPEB5A)
- Sequoia Capital (UCWrF0oN6unbXrWsTN7RctTw)
... and 7 more channels


## Fetch videos for each business-cluster channel
Retrieves videos from Finder's `videos` collection for all channels in the business cluster and groups them by `channelId`.


This cell fetches all videos for the business-cluster channels and groups them by `channelId`. It also prints quick per-channel diagnostics to verify data coverage.


In [18]:
from collections import defaultdict

if not channel_ids:
    print('No channel ids available to query videos.')
    videos_by_channel = {}
else:
    cursor = videos_col.find(
        {'channelId': {'$in': channel_ids}},
        {
            '_id': 0,
            'videoId': 1,
            'channelId': 1,
            'title': 1,
            'publishedAt': 1,
            'viewCount': 1,
            'likeCount': 1,
            'commentCount': 1,
            'createdAt': 1,
            'created_at': 1,
            'dateCreated': 1,
            'insertedAt': 1
        }
    ).sort([('channelId', 1), ('publishedAt', -1)])

    videos_by_channel = defaultdict(list)
    for doc in cursor:
        videos_by_channel[doc['channelId']].append(doc)

    print(f'Fetched videos for {len(videos_by_channel)} channels.')
    total_videos = sum(len(v) for v in videos_by_channel.values())
    print(f'Total videos fetched: {total_videos}')

    for channel in business_channels:
        cid = channel.get('channelId')
        title = channel.get('title', '(untitled)')
        channel_videos = videos_by_channel.get(cid, [])
        print(f'{title} ({cid}) -> {len(channel_videos)} videos')

        for video in channel_videos[:1]:
            print(
                f"  • {video.get('title', '(no title)')} "
                f"| views={video.get('viewCount')} | likes={video.get('likeCount')} | comments={video.get('commentCount')} "
                f"| published={video.get('publishedAt')}"
            )


Fetched videos for 27 channels.
Total videos fetched: 1344
Bg2 Pod (UC-yRDvpR99LUc5l7i7jLzew) -> 44 videos
  • ChatGPT – The Super Assistant Era | BG2 Guest Interview | views=7574 | published=2026-03-15 16:25:13
The Prof G Pod – Scott Galloway (UC1E1SVcVyU3ntWMSQEp38Yw) -> 50 videos
  • The Hidden Engine of China’s AI Boom | China Decode | views=17697 | published=2026-04-21 08:00:53
Asianometry (UC1LpsuAUaKoMzzJSEt5WImw) -> 50 videos
  • How To Test 208 Billion Transistors | views=149040 | published=2026-04-12 23:00:26
Lenny's Podcast (UC6t1O76G0jYXOAoYCm153dA) -> 50 videos
  • Head of Growth (Anthropic):  “Claude is growing itself at this point” | views=77764 | published=2026-04-05 12:30:57
a16z (UC9cn0TuPq4dnbTY-CBsm8XA) -> 50 videos
  • The Era of AI Agents | Aaron Levie on The a16z Show | views=19976 | published=2026-04-08 14:30:00
Dan Martell (UCA-mWX9CvCTVFWRMb9bKc9w) -> 50 videos
  • How to Make Time For Everything (Seriously, everything) | views=63671 | published=2026-04-20 13:

## Build a tabular video snapshot
This cell flattens grouped videos into a DataFrame so subsequent joins, metrics, and artifact exports can use a consistent tabular schema.


In [5]:
# Optional: flatten all fetched videos to a DataFrame for downstream analysis/export
import pandas as pd

all_videos = [v for vids in videos_by_channel.values() for v in vids] if videos_by_channel else []
videos_df = pd.DataFrame(all_videos)
print('videos_df shape:', videos_df.shape)
videos_df.head(10)


videos_df shape: (1344, 5)


,videoId,channelId,publishedAt,title,viewCount
0,MIKej1HCRW0,UC-yRDvpR99LUc5l7i7jLzew,2026-03-15 16:25:13,ChatGPT – The Super Assistant Era | BG2 Guest ...,7574
1,jA8ZQfq_Hzs,UC-yRDvpR99LUc5l7i7jLzew,2025-12-23 23:15:04,AI Enterprise - Databricks & Glean | BG2 Guest...,24210
2,Gnl833wXRz0,UC-yRDvpR99LUc5l7i7jLzew,2025-10-31 23:17:31,All things AI w @altcap @sama & @satyanadella....,228167
3,KX6q6lvoYtM,UC-yRDvpR99LUc5l7i7jLzew,2025-10-14 23:56:17,"AI Bubble, Stablecoin Boom, and Runnin' Down a...",71463
4,pE6sw_E9Gh0,UC-yRDvpR99LUc5l7i7jLzew,2025-09-26 03:36:04,"NVIDIA: OpenAI, Future of Compute, and the Ame...",426866
5,yLTSqBzKG2s,UC-yRDvpR99LUc5l7i7jLzew,2025-09-11 21:25:36,Inside OpenAI Enterprise: Forward Deployed Eng...,23451
6,hUJz55AsUz4,UC-yRDvpR99LUc5l7i7jLzew,2025-08-28 23:51:07,"China, China, China. Breaking Down China’s Tec...",114510
7,fTqINzeudJ4,UC-yRDvpR99LUc5l7i7jLzew,2025-07-31 15:25:00,"China Open-Source, Compute Arms Race, Reorderi...",54888
8,X52BNWZrXSk,UC-yRDvpR99LUc5l7i7jLzew,2025-07-10 23:23:42,"Michael Dell – Invest America Act Becomes Law,...",75922
9,IOoRXSyezBE,UC-yRDvpR99LUc5l7i7jLzew,2025-06-21 16:36:30,Coatue Pt2. Open AI’s Kevin Weil Dives into Al...,42867


## Pinecone embedding workflow (reused fetch-or-embed pattern)
This section reuses the same **fetch then embed-if-missing** design used in `Embeddings-Graph.ipynb` (`get_or_create_channel_description_embeddings`) and generalizes it into a helper for both:
- channel descriptions (`ChannelDescriptions` namespace)
- video titles (`VideoTitles` namespace)

That gives idempotent runs: existing vectors are fetched, and only missing records are embedded/upserted.


This cell authenticates against Pinecone and opens the `finder` index. The resulting client and index objects are reused by the embedding fetch-or-create workflow.


In [6]:
from pinecone import Pinecone

try:
    pinecone_api_key = userdata.get('PINECONE_API_KEY')
    pc = Pinecone(api_key=pinecone_api_key)
    pinecone_index = pc.Index('finder')
    print('✅ Connected to Pinecone index: finder')
except Exception as e:
    print(f'❌ Pinecone connection failed: {e}')


✅ Connected to Pinecone index: finder


## Define reusable embedding fetch-or-create helper
This helper first fetches existing vectors, embeds only missing texts, upserts them, and returns a complete in-memory embedding dictionary for downstream analysis.


In [7]:
import numpy as np

def fetch_or_embed_texts(
    pc_client,
    index,
    namespace,
    id_to_text,
    model="multilingual-e5-large",
    batch_size=96
):
    """
    1) Fetch existing vectors by id from Pinecone.
    2) Embed missing texts.
    3) Upsert missing vectors.
    4) Return dict[str, list[float]].
    """

    # Normalize keys once, so ids and text lookups stay consistent
    normalized = {
        str(k): v for k, v in id_to_text.items()
        if v is not None and str(v).strip() != ""
    }

    ids = list(normalized.keys())
    embeddings = {}

    # Fetch existing vectors
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i + batch_size]
        response = index.fetch(ids=batch_ids, namespace=namespace)
        vectors = getattr(response, "vectors", {}) or {}

        for rid, payload in vectors.items():
            values = payload.get("values") if isinstance(payload, dict) else getattr(payload, "values", None)
            if values is not None:
                embeddings[rid] = values

    # Embed missing vectors
    missing_ids = [rid for rid in ids if rid not in embeddings]

    for i in range(0, len(missing_ids), batch_size):
        batch_missing = missing_ids[i:i + batch_size]
        input_texts = [normalized[rid] for rid in batch_missing]

        embed_resp = pc_client.inference.embed(
            model=model,
            inputs=input_texts,
            parameters={"input_type": "passage"}
        )

        rows = getattr(embed_resp, "data", embed_resp)

        to_upsert = []
        for rid, row in zip(batch_missing, rows):
            # Pinecone returns "values", not "embedding"
            values = row.get("values") if isinstance(row, dict) else getattr(row, "values", None)
            if values is None:
                raise ValueError(f"No embedding values returned for id={rid}")

            embeddings[rid] = values
            to_upsert.append({
                "id": rid,
                "values": values,
                "metadata": {"text": normalized[rid][:1000]}
            })

        if to_upsert:
            index.upsert(vectors=to_upsert, namespace=namespace)

    return embeddings


## Prepare channel-description and video-title embedding workloads
This cell builds clean text dictionaries for channel descriptions and video titles, filtering out empty records before embedding retrieval.


In [8]:
# Build text dictionaries for embedding workloads
channel_desc_text = {
    str(ch.get('channelId')): (ch.get('description') or '').strip()
    for ch in business_channels
    if ch.get('channelId') and (ch.get('description') or '').strip()
}

video_title_text = {}
for cid, vids in videos_by_channel.items():
    for v in vids:
        vid = str(v.get('videoId')) if v.get('videoId') else None
        title = (v.get('title') or '').strip()
        if vid and title:
            video_title_text[vid] = title

print(f'Channels with descriptions: {len(channel_desc_text)}')
print(f'Videos with titles: {len(video_title_text)}')


Channels with descriptions: 27
Videos with titles: 1344


## Materialize embeddings for channels and videos
This cell executes the fetch-or-embed flow for both namespaces and reports final embedding coverage used in the distance analysis.


In [9]:
channel_desc_embeddings = fetch_or_embed_texts(
    pc_client=pc,
    index=pinecone_index,
    namespace='ChannelDescriptions',
    id_to_text=channel_desc_text,
)

video_title_embeddings = fetch_or_embed_texts(
    pc_client=pc,
    index=pinecone_index,
    namespace='VideoTitles',
    id_to_text=video_title_text,
)

print(f'Channel-description embeddings available: {len(channel_desc_embeddings)}')
print(f'Video-title embeddings available: {len(video_title_embeddings)}')


Channel-description embeddings available: 27
Video-title embeddings available: 1344


## Reduce video embeddings to 20 dimensions and build export dataset
This section removes downstream analysis and only prepares a compact export table. It applies PCA (20 components) over the entire set of available video-title embeddings, then joins engagement metrics and date fields from Finder videos.


In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

all_videos = [v for vids in videos_by_channel.values() for v in vids] if videos_by_channel else []
videos_df = pd.DataFrame(all_videos)

# Keep only rows that have an embedding available
records = []
for _, row in videos_df.iterrows():
    video_id = str(row.get('videoId')) if pd.notna(row.get('videoId')) else None
    emb = video_title_embeddings.get(video_id) if video_id else None
    if emb is None:
        continue
    records.append({
        'video_id': video_id,
        'channel_id': row.get('channelId'),
        'video_title': row.get('title'),
        'view_count': pd.to_numeric(row.get('viewCount'), errors='coerce'),
        'like_count': pd.to_numeric(row.get('likeCount'), errors='coerce'),
        'comment_count': pd.to_numeric(row.get('commentCount'), errors='coerce'),
        'date_created_1': row.get('createdAt') if pd.notna(row.get('createdAt')) else row.get('created_at'),
        'date_created_2': row.get('dateCreated') if pd.notna(row.get('dateCreated')) else row.get('insertedAt'),
        'date_published': row.get('publishedAt'),
        'embedding': np.asarray(emb, dtype=float),
    })

export_df = pd.DataFrame(records)
print('Rows with embeddings:', export_df.shape[0])

if export_df.empty:
    reduced_df = export_df.drop(columns=['embedding'], errors='ignore').copy()
else:
    emb_matrix = np.vstack(export_df['embedding'].values)
    n_components = min(20, emb_matrix.shape[0], emb_matrix.shape[1])
    pca = PCA(n_components=n_components, random_state=42)
    reduced = pca.fit_transform(emb_matrix)

    reduced_cols = [f'embedding_reduced_{i+1:02d}' for i in range(n_components)]
    reduced_df = export_df.drop(columns=['embedding']).reset_index(drop=True)
    reduced_df[reduced_cols] = reduced

    # Guarantee exactly 20 export columns for reduced embedding
    for i in range(n_components, 20):
        reduced_df[f'embedding_reduced_{i+1:02d}'] = np.nan

reduced_df.head(10)


## Export reduced video embeddings artifact
Writes a single CSV with reduced embeddings and metadata to Google Drive.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

artifact_root = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest')
artifact_root.mkdir(parents=True, exist_ok=True)

export_csv_path = artifact_root / 'business_cluster_video_embeddings_reduced_20d.csv'
reduced_df.to_csv(export_csv_path, index=False)

print('Saved export:')
print(f' - {export_csv_path}')
print(f' - rows: {len(reduced_df)}')
print(f' - columns: {len(reduced_df.columns)}')


## Output schema
The exported CSV includes: `video_id`, `channel_id`, `video_title`, `view_count`, `like_count`, `comment_count`, `date_created_1`, `date_created_2`, `date_published`, and `embedding_reduced_01` ... `embedding_reduced_20`.
